# 03. Categorical Feature Engineering
### Notebook 3 – Categorical Feature Engineering

Dataset: Online Retail Transactions (`data.csv`)


In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('data.csv', encoding='ISO-8859-1')
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34


# 1. Categorical Variables

A categorical variable represents data that takes on a limited, fixed set of values, such as a country name or a product code, rather than a continuous numeric range.

**When to use:** applies whenever a column represents a label, group, or category rather than a measurable quantity.

**Why:** most machine learning models require numeric input, so categorical variables must be encoded before they can be used, and the right encoding depends on the type of category.

In [2]:
df['Country'].dtype
df['Country'].unique()[:10]

<ArrowStringArray>
['United Kingdom',         'France',      'Australia',    'Netherlands',
        'Germany',         'Norway',           'EIRE',    'Switzerland',
          'Spain',         'Poland']
Length: 10, dtype: str

# 2. Nominal Variables

A nominal variable has categories with no inherent order or ranking between them.

**When to use:** applies to variables like `Country` or `StockCode`, where one category is not naturally greater or smaller than another.

**Why:** encoding nominal variables with a scheme that implies order, such as plain integer labels, can mislead models that assume numeric relationships between values.

In [3]:
df['Country'].value_counts().head()

Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Name: count, dtype: int64

# 3. Ordinal Variables

An ordinal variable has categories with a meaningful order, but the spacing between categories is not necessarily equal.

**When to use:** applies to variables like a customer tier or a satisfaction rating, where categories can be ranked but not treated as equally spaced numbers.

**Why:** encoding the order correctly allows models to use the ranking information, while treating it as nominal would throw away that structure.

In [4]:
spend_rank = df.groupby('CustomerID')['TotalPrice'].sum()
spend_tier = pd.qcut(spend_rank, q=3, labels=['Low', 'Medium', 'High'])
spend_tier.head()

CustomerID
12346.0     Low
12347.0    High
12348.0    High
12349.0    High
12350.0     Low
Name: TotalPrice, dtype: category
Categories (3, str): ['Low' < 'Medium' < 'High']

# 4. Binary Variables

A binary variable has exactly two possible categories, often represented as 0 and 1.

**When to use:** applies to variables like whether a transaction is a return or not.

**Why:** binary variables can be encoded directly as a single 0 or 1 column without needing multiple columns, keeping the feature space compact.

In [5]:
df['IsReturn'] = (df['Quantity'] < 0).astype(int)
df[['Quantity', 'IsReturn']].head()

,Quantity,IsReturn
0,6,0
1,6,0
2,8,0
3,6,0
4,6,0


# 5. One-Hot Encoding

One-hot encoding creates a separate binary column for each category, marking presence with 1 and absence with 0.

**When to use:** best suited for nominal variables with a small to moderate number of unique categories.

**Why:** it avoids implying any order between categories, but it increases the number of columns significantly as the number of unique categories grows.

In [6]:
top_countries = df['Country'].value_counts().nlargest(5).index
df['Country_grouped'] = np.where(df['Country'].isin(top_countries), df['Country'], 'Other')
one_hot = pd.get_dummies(df['Country_grouped'], prefix='Country')
one_hot.head()

,Country_EIRE,Country_France,Country_Germany,Country_Other,Country_Spain,Country_United Kingdom
0,False,False,False,False,False,True
1,False,False,False,False,False,True
2,False,False,False,False,False,True
3,False,False,False,False,False,True
4,False,False,False,False,False,True


# 6. Ordinal Encoding

Ordinal encoding assigns each category an integer value that reflects its rank order.

**When to use:** best suited for ordinal variables where the categories have a natural, meaningful order.

**Why:** it preserves the ranking information in a single compact column, but should not be used on nominal variables since it would falsely imply an order.

In [7]:
tier_order = {'Low': 0, 'Medium': 1, 'High': 2}
df_customer = spend_tier.map(tier_order)
df_customer.head()

CustomerID
12346.0    0
12347.0    2
12348.0    2
12349.0    2
12350.0    0
Name: TotalPrice, dtype: category
Categories (3, int64): [0 < 1 < 2]

# 7. Label Encoding

Label encoding assigns each unique category an arbitrary integer, without any regard to order.

**When to use:** commonly used for nominal variables when working with tree based models that can handle arbitrary integer splits without assuming order.

**Why:** it is compact and simple, but linear or distance based models can misinterpret the arbitrary integers as having a meaningful order or magnitude.

In [8]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['StockCode_label'] = le.fit_transform(df['StockCode'])
df[['StockCode', 'StockCode_label']].head()

,StockCode,StockCode_label
0,85123A,3536
1,71053,2794
2,84406B,3044
3,84029G,2985
4,84029E,2984


# 8. Frequency Encoding

Frequency encoding replaces each category with the number or proportion of times it occurs in the dataset.

**When to use:** useful for nominal variables with many categories, especially when the frequency of a category itself carries predictive signal.

**Why:** it keeps the feature space to a single column regardless of cardinality, and can capture useful signal, but categories with the same frequency become indistinguishable.

freq = df['StockCode'].value_counts()
df['StockCode_freq'] = df['StockCode'].map(freq)
df[['StockCode', 'StockCode_freq']].head()

# 9. Target Encoding

Target encoding replaces each category with a statistic of the target variable computed for that category, such as the mean.

**When to use:** useful for high cardinality nominal variables in supervised learning problems, where a strong relationship exists between the category and the target.

**Why:** it can capture strong predictive signal in a single column, but is prone to overfitting and target leakage if not computed carefully using cross validation or hold out schemes.

In [9]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
country_target_mean = df.groupby('Country')['TotalPrice'].mean()
df['Country_target_enc'] = df['Country'].map(country_target_mean)
df[['Country', 'TotalPrice', 'Country_target_enc']].head()

,Country,TotalPrice,Country_target_enc
0,United Kingdom,15.30,16.525065
1,United Kingdom,20.34,16.525065
2,United Kingdom,22.00,16.525065
3,United Kingdom,20.34,16.525065
4,United Kingdom,20.34,16.525065


# 10. Rare Category Grouping

Rare category grouping combines infrequent categories into a single group, often labeled as `Other`.

**When to use:** applies when a categorical variable has many categories that each occur very few times in the data.

**Why:** rare categories provide little statistical signal individually and can cause overfitting or unstable encodings, so grouping them stabilizes the feature.

In [10]:
country_counts = df['Country'].value_counts()
rare_countries = country_counts[country_counts < 100].index
df['Country_grouped_rare'] = df['Country'].replace(rare_countries, 'Other')
df['Country_grouped_rare'].value_counts()

Country_grouped_rare
United Kingdom     495478
Germany              9495
France               8557
EIRE                 8196
Spain                2533
Netherlands          2371
Belgium              2069
Switzerland          2002
Portugal             1519
Australia            1259
Norway               1086
Italy                 803
Channel Islands       758
Finland               695
Cyprus                622
Sweden                462
Unspecified           446
Austria               401
Denmark               389
Other                 358
Japan                 358
Poland                341
Israel                297
USA                   291
Hong Kong             288
Singapore             229
Iceland               182
Canada                151
Greece                146
Malta                 127
Name: count, dtype: int64

# 11. Category Cardinality

Cardinality refers to the number of unique categories in a categorical variable.

**When to use:** this is not a technique itself but a property to check before choosing an encoding method, since low and high cardinality variables need different treatment.

**Why:** low cardinality variables work well with one-hot encoding, while high cardinality variables usually require frequency, target, or embedding based encoding to avoid an explosion of columns.

In [11]:
df[['Country', 'StockCode', 'Description']].nunique()

Country          38
StockCode      4070
Description    4223
dtype: int64

# 12. Handling Unknown Categories

This refers to the strategy used when a category appears at prediction time that was never seen during training.

**When to use:** must be planned for any encoding scheme applied to data that will later be used for scoring new, unseen data.

**Why:** without a defined fallback, such as mapping unseen categories to `Other` or a default value, the pipeline can fail or silently produce invalid encodings on new data.

In [12]:
known_countries = set(df['Country'].unique())
new_sample = pd.Series(['United Kingdom', 'Atlantis'])
new_sample_encoded = new_sample.where(new_sample.isin(known_countries), 'Unknown')
new_sample_encoded

0    United Kingdom
1           Unknown
dtype: str

# 13. Combining Categories

Combining categories means merging two or more categorical columns, or grouping categories within one column, based on domain logic rather than just frequency.

**When to use:** applies when separate categories represent essentially the same underlying concept, or when interactions between two categorical columns are meaningful.

**Why:** it reduces redundant distinctions in the data and can create more meaningful groupings than either column provides alone.

In [13]:
df['Country_StockPrefix'] = df['Country'] + '_' + df['StockCode'].str[0]
df['Country_StockPrefix'].value_counts().head()

Country_StockPrefix
United Kingdom_2    399003
United Kingdom_8     59610
United Kingdom_4     10719
Germany_2             8124
France_2              7532
Name: count, dtype: int64